---
title: Elements of Computations
abstract: |
    Motivated by practical problems such as computing the greatest common divisor (GCD) and the inverse square root, this notebook introduces the fundamental elements of computation in C++. It explores C++ syntax for declaring variables and representing values of various types, including integers, floating-point numbers, characters, strings, and pointers. The design of C++ is compared with that of Python to highlight key differences in type systems, memory management, and operator behavior across programming languages.
execute:
  skip: true
---

In [ ]:
from __init__ import *

## Motivations

This notebook introduces the basic ingredients for computations in C++ and explores their limitations. To provide motivation for the subject, we begin by presenting two problems below.

### GCD

Consider the problem of computing the [greatest common divisor (GCD)](https://en.wikipedia.org/wiki/Greatest_common_divisor), which is a fundamental concept in number theory.[^Galois] 

::::{prf:definition} GCD
:label: def:gcd

The GCD of two non-zero integers $a$ and $b$, denoted as $\operatorname{gcd}(a, b)$  is *the largest integer $d$ that divides both $a$ and $b$*, i.e., $d|a$ and $d|b$.


::::

[^Galois]: For instance, two numbers with GCD equal to 1 are called coprime, which is a concept used in [finite fields](https://en.wikipedia.org/wiki/Finite_field) with significant applications to different areas such as coding theory and cryptography.

The following is an implementation in C++.

1. To specify and store the non-zero integers $a$ and $b$:

In [ ]:
%%cpp
int a=2*3*4, b=3*4*5; // integer variables declaration and initialization
cout << format("gcd({}, {})=?\n", a, b); // print formatted string

2. To compute the GCD of $a$ and $b$ step-by-step, repeatedly run the following program until *`a` becomes `0`, in which case the absolute value of `b` is the gcd*.
  ::::{code} cpp
  :label: code_gcd1
  :caption: Computation of the GCD of the non-zero integers `a` and `b`.
  :linenos:
  {
      b = b % a; // assignment and modulo operation
      int c = b; // variable in the block scope of the compound statement
      b = a;
      a = c;
  } // compound statement
  ::::

In [ ]:
%%cpp
{
    b = b % a;
    int c = b;
    b = a;
    a = c;
    cout << format("gcd({}, {})\n", a, b); // intermediate answer
} // compound statement

::::{caution}

Want to see a crash? Run the code further after getting the GCD from the printout `gcd(0, ...)`! Press <kbd>0, 0</kbd> to restart the kernel, and then run all above cells again.

::::

::::{exercise}
:label: ex:gcd1

How can the above program fail to give the correct GCD?

:::{hint}
:class: dropdown

The code fails for some choices of $a$ and $b$.

:::

::::

YOUR ANSWER HERE

If you can answer the above questions, great! But don't worry if you cannot. We will learn some basic programming elements to solve [](#ex:gcd1) in this notebook.

::::{exercise}
:label: ex:gcd2

Explain how the program computes the GCD. Evaluate the effectiveness of the algorithm and suggest improvements.

:::{hint}
:class: dropdown

The program uses the [](https://en.wikipedia.org/wiki/Euclidean_algorithm). Compare it with [Python's implementation for the `gcd` function in C](https://github.com/python/cpython/blob/47bc10e6b3cb44658da275f3484781ef2a2b9222/Objects/longobject.c#L5726C1-L5931C2), applying [Lehmer's GCD algorithm](https://en.wikipedia.org/wiki/Lehmer%27s_GCD_algorithm). See also the [GitHub issue](https://github.com/python/cpython/issues/66676) for the implementation detail.

If you would like to give the computational complexity, the worst case performance is when $a$ and $b$ are two consecutive [Fibonacci numbers](https://en.wikipedia.org/wiki/Fibonacci_sequence), such as

> `a = 8` ($F_6$) and `b = 13` ($F_7$).

$F_n$ grows exponentially as $O(\phi^n)$ where $\phi$ is the golden ratio.

:::

::::

YOUR ANSWER HERE

[](#ex:gcd2) is more challenging, as it requires quite a bit of [computational thinking](https://en.wikipedia.org/wiki/Computational_thinking) that you will develop over time, mostly outside the classroom. Dive into it with the help of AI:

In [ ]:
%%hermes
What is the fastest possible algorithm for computing GCD?

::::{admonition} Hermes
:class: dropdown

There is no single "fastest" GCD — it depends on the inputs. The practical answer for machine-sized integers is **Stein's binary GCD** (1961), which avoids the expensive division and uses only bit shifts, subtractions, and comparisons. For very large (big-integer) inputs, **Lehmer's / half-gcd** algorithms are asymptotically faster.

**Euclid's algorithm** (the lecture's approach) is already optimal in the *number of iterations*: it takes $O(\log(\min(a,b)))$ steps, and the worst case is exactly consecutive Fibonacci numbers — the $F_{n-1}, F_n$ case analyzed in [](#ex:gcd2). Each step, however, performs a division, the slowest integer instruction the CPU decodes.

**Binary GCD** produces the same answer but never divides:
1. It strips out all common factors of 2, 
2. strips the remaining factors of 2 from one operand, then 
3. repeatedly replaces the larger of the two odd numbers with their difference. 
    
Every operation is a cheap shift or subtract, which is why it is typically the fastest GCD in practice and what many libraries use.

```cpp
long long gcd_binary(long long a, long long b) {
    if (a == 0) { return b; }
    if (b == 0) { return a; }
    int shift = 0;
    while (((a | b) & 1) == 0) { a >>= 1; b >>= 1; ++shift; } // common 2s
    while ((a & 1) == 0) { a >>= 1; }                          // 2s out of a
    do {
        while ((b & 1) == 0) { b >>= 1; }
        if (a > b) { long long t = a; a = b; b = t; }
        b -= a;
    } while (b != 0);
    return a << shift;
}
```

Design notes:
- **No division at all** — the correctness rests on the identity that $\gcd(2a, 2b) = 2\,\gcd(a,b)$ and $\gcd(2a, b) = \gcd(a, b)$ when $b$ is odd, so factors of 2 can be deferred and shifted back at the end.
- **Same complexity class, smaller constant** — still $O(\log(\min(a,b)))$ iterations, but each iteration is a cheap shift/subtract instead of a division, so the measured runtime is far lower (see the timing cell above).
- `gcd_euclid` is kept for comparison so you can see both return identical values on the Fibonacci worst case.

::::

The following benmarks different GCD algorithms in Python:

In [ ]:
from gcd import *

# Consecutive Fibonacci numbers
n = 100_000
f1 = fib(n)
f2 = fib(n - 1)

# warmup
gcd_euclid(f1, f2)
gcd_binary(f1, f2)
gcd(f1, f2)

t0 = perf_counter()
g1 = gcd_euclid(f1, f2)
t1 = perf_counter()

g2 = gcd_binary(f1, f2)
t2 = perf_counter()

g3 = gcd(f1, f2)
t3 = perf_counter()

print("Euclid  :", t1 - t0, "seconds")
print("Binary  :", t2 - t1, "seconds")
print("Library :", t3 - t2, "seconds")
print("GCDs equal:", g1 == g2 == g3)

### Inverse Square Root

While computing the greatest common divisor (GCD) involves only integers, many computations involve rational, real numbers, and even complex numbers. An example is to compute the *inverse/reciprocal square root* 

$$
\begin{align}
\operatorname{rsqrt}(x)&:=\frac{1}{\sqrt{x}} && \text{for $x>0$},
\end{align}
$$ (eq:rsqrt)

which is an important computation for 3D graphics, e.g., in first-person-shooting games such as [](https://en.wikipedia.org/wiki/Quake_III_Arena).

The following is an implementation in C++.

1. Store the real number $x$ *approximately* as a floating-point number, and compute its reciprocal $y=\frac1x$ as the initial guess of the inverse square root:

In [ ]:
%%cpp
double x = 10./3;  // a floating point number as input
cout << format("rsqrt({})=?\n", x);

auto y = 1/x;      // reciprocal, type `auto`matically deduced
cout << format("Initial guess: {}\n", y);

2. To compute the desired square root of the reciprocal, repeatedly runs the following until the answer remains unchanged/steady.
  ::::{code} cpp
  :label: code_inv_sqrt1
  :caption: Update rule for the inverse square root of `x`.
  :linenos:
  y = (y+1/y/x)/2;   // update the answer
  ::::

In [ ]:
%%cpp
y = (y+1/y/x)/2;
cout << format("Intermediate answer: {}\n", y);

You can verify whether the answer is correct by recovering $x$ from $y$ as $\frac{1}{y^2}$:

In [ ]:
%%cpp
1/y/y

The program [can be improved](https://mrober.io/papers/rsqrt.pdf) by:

1. A better initial guess
  ::::{code} cpp
  :label: code_fast_inv_sqrt1_guess
  :caption: Initial guess for the fast inverse square root of `x`.
  :linenos:
   auto i = *reinterpret_cast<int64_t *>(&x);   // copy x to i as an integer
   i = 0x5fe6eb50c7b537a9 - (i >> 1);           // ???
   auto y = *reinterpret_cast<double *>(&i);    // copy i to y as a double
  ::::

In [ ]:
%%cpp
double x = 10./3;  // input
cout << format("rsqrt({})=?\n", x);

auto i = *reinterpret_cast<int64_t *>(&x);
i = 0x5fe6eb50c7b537a9 - (i >> 1);
auto y = *reinterpret_cast<double *>(&i);
cout << format("Initial guess: {}\n", y);  // initial guess

2. An alternative update rule
  ::::{code} cpp
  :label: code_fast_inv_sqrt1_update
  :caption: Update rule for the fast inverse square root of `x`.
  :linenos:
  y = y * (1.5 - (x*0.5 * y * y));
  ::::

In [ ]:
%%cpp
y = y * (1.5 - (x/2 * y * y));
cout << format("Intermediate answer: {}\n", y);

In [ ]:
%%cpp
1/y/y

::::{exercise}
:label: ex:inv_sqrt1

How can the above program fail to give the inverse square root?

:::{hint}
:class: dropdown

The code fails for some choices of $x$.

:::

::::

YOUR ANSWER HERE

::::::{exercise}
:label: ex:inv_sqrt2

Explain how the programs compute the inverse square root.

:::::{hint}
:class: dropdown

With $y:=\frac{1}{\sqrt{x}}$, we can obtain an equation in $y$ whose root is the desired inverse square root:
\begin{align}
f(y):=y^2 - \frac1x
\end{align}

The question is how to compute the root of $f(y)$. To understand the improved initial guess, see a related video below:

::::{card}
:header: The Fast Inverse Square Root -- 0x5f3759df explained!!
:footer: [open in new tab](https://www.youtube.com/embed/NCuf2tjUsAY?si=oaqOOD64Z6kxCS_m)

:::{iframe} https://www.youtube.com/embed/NCuf2tjUsAY?si=oaqOOD64Z6kxCS_m
:width: 100%
:::
::::

:::::

::::::

YOUR ANSWER HERE

Answering both [](#ex:inv_sqrt1) and [](#ex:inv_sqrt2) requires a very good understanding of how real numbers are represented in computers in addition to the numerical methods involved:

In [ ]:
%%hermes
List different methods of computing the square root using basic arithmetic 
operations.

::::{admonition} Hermes
:class: dropdown

Computing $\sqrt{x}$ with only $+$, $-$, $\times$, $\div$ (and comparisons) is a classic. The lecture already met one such method in *disguise* — the fast inverse-square-root update rule $y \leftarrow y(1.5 - x y^2/2)$ is just Newton's method applied to $f(y)=1/y^2 - x$. The three methods below are its companions; **all three use nothing but the four basic operations.**

**1. Bisection (false position, "halve the bracket").** Keep a range $[lo,hi]$ with $lo^2 \le x \le hi^2$; put $mid=(lo+hi)/2$, and if $mid^2>x$ the root is in $[lo,mid]$, else in $[mid,hi]$. Repeat until the bracket is tiny. Uses only a multiply and a compare per step ($mid^2$ vs $x$) — no division. Converges **linearly** (one extra correct digit per step), so it is reliable but slow.

**2. Newton–Raphson (the Babylonian / "heron" method).** Solve $f(y)=y^2-x=0$; Newton's step $y \leftarrow y - f/f'$ with $f'=2y$ is exactly
$$y_{k+1} = \frac{y_k + x/y_k}{2}.$$
It needs a division and an average per step, and it converges **quadratically**: the error roughly *squares* each iteration, so 4–5 steps give full double precision from a crude start. This is the same update family as the lecture's $y \leftarrow y(1.5 - xy^2/2)$, which is Newton on $1/y^2 = x$. In practice this is the family $\sqrt{x}$ hardware and libraries build on.

**3. Digit-by-digit ("schoolbook" / long-division) extraction.** The manual method: group the digits in pairs from the decimal point; at each step, with current answer $p$ and remainder $r$, find the largest digit $d$ such that $(20p+d) d \le r$. It is the square-root analogue of long division and uses **no division at all** in the digit search — only $+$, $-$, $\times$ and comparisons — which is exactly why it was the go-to method before scientific calculators.

The cell above runs all three on the lecture's $x=10/3$ (plus a digit check on $1056$) and prints Newton's converging steps.

Design notes:
- **Same answer, different speeds.** Bisection and Newton both find $\sqrt{10/3} \approx 1.825741858350554$; Newton reaches full precision in ~5 iterations where bisection would need ~50. The digit method is exact for perfect squares and gives $\lfloor\sqrt{x}\rfloor$ otherwise.
- **Newton is the bridge to the lecture.** Its update $y \leftarrow y(1.5 - xy^2/2)$ for $1/\sqrt{x}$ is the same contraction, just targeting the reciprocal — which is why a good *initial guess* (the famous bit-trick) matters so much there.
- **All arithmetic, no `sqrt()`.** The whole point is these reproduce the result of the library call using only the basic operations the CPU is guaranteed to have.

::::

The following benmarks different inverse square root algorithms in Python:

In [ ]:
from rsqrt import *

x = 10.0 / 3.0
print(f"x = {x}")

print(f"bisect : {rsqrt_bisect(x):.15f}")
print(f"newton : {rsqrt_newton(x):.15f}")
print(f"math   : {1/sqrt(x):.15f}")
print(
    f"digits : rsqrt(1056) ~= "
    f"{rsqrt_digits(1056):.15f}"
)

REPEAT = 100000

# warmup
rsqrt_bisect(x)
rsqrt_newton(x, verbose=False)
1.0 / sqrt(x)

t0 = perf_counter()
for _ in range(REPEAT):
    rsqrt_bisect(x)
t1 = perf_counter()

for _ in range(REPEAT):
    rsqrt_newton(x, verbose=False)
t2 = perf_counter()

for _ in range(REPEAT):
    1.0 / sqrt(x)
t3 = perf_counter()

print(f"Bisection : {(t1-t0)/REPEAT:.3e} s")
print(f"Newton    : {(t2-t1)/REPEAT:.3e} s")
print(f"Library   : {(t3-t2)/REPEAT:.3e} s")

In the sequel, we will tackle the easier task of learning the elements of C++ involved in carrying out the above computations.

## Integer

### Literal

How to specify an integer value in C++?

One way to specify a value is to use a *literal*, which is a fixed value directly embedded in the source code. A literal can be written in different ways:

In [ ]:
%%cpp
15 // in decimal

In [ ]:
%%cpp
0b1111 // in binary

In [ ]:
%%cpp
017 // in octal

In [ ]:
%%cpp
0xF // in hexadecimal

All of the above have the same value of type `int`!

In comparison, Python also uses the same syntax except for octal format:

In [ ]:
15, 0b1111, 0o17, 0xF

C++ also supports [different types of integer literals of different sizes](https://en.cppreference.com/w/cpp/language/types.html), e.g., 3 billions is represented as type `long`:

In [ ]:
%%cpp
3000000000

$10^{19}$ is represented as unsigned integer type:

In [ ]:
%%cpp
10000000000000000000 // with a warning

We can use the [appropriate suffixes](https://en.cppreference.com/w/cpp/language/integer_literal.html) to specify the desired type:

In [ ]:
%%cpp
10000000000000000000uLL // no warning message

In [ ]:
%%cpp
1L // not the default type `int`.

The size of different types can be obtained by the [`sizeof` operator](https://en.cppreference.com/w/cpp/language/sizeof.html)[^sizeof]:

[^sizeof]: `sizeof` is not a function, even though its usage—such as sizeof(long long)f—resembles a function call. In C++, You cannot pass a type as an argument to a regular function. For template functions, types can be passed but using angle brackets instead of parentheses.

In [ ]:
%%cpp
sizeof(short)  // in the unit of byte (8 bits)

In [ ]:
%%cpp
sizeof(int)

In [ ]:
%%cpp
sizeof(long)

In [ ]:
%%cpp
sizeof(long long)

Let's calculate the number of values of type `int`:

In [ ]:
%%cpp
1uLL << sizeof(int)*8

::::{caution} Shouldn't we use `cout <<` instead to print the number of values?
:class: dropdown

The number of values of type `int` can be computed using the [bitwise shift operator `<<`](https://en.cppreference.com/w/cpp/language/operator_arithmetic.html#Bitwise_shift_operators), which is not the usual [stream insertion operator](https://en.cppreference.com/w/cpp/io/basic_ostream/operator_ltlt2.html) used to print to standard output/error `cout`/`cerr`:[^bitwise]

::::

[^bitwise]: Bitwise operators perform operations on the bits of the binary representation of integers. You may explore other bitwise operators to learn about them:
    1. Bitwise Shift: `<<`, `>>`
    2. Bitwise AND: `&`
    3. Bitwise XOR: `^`
    4. Bitwise OR: `|`

::::{caution} What happens if you change `1uLL` to `1`?
:class: dropdown

The computed value will be wrong. You may see a warning such as

```
warning: shift count >= width of type [-Wshift-count-overflow]
```

because `int` is not large enough to represent the number of values of type `int`! Can you explain what the computed value actually is? Try running the code in `xeus-cpp` kernel as a comparison.

::::

We can also obtain the exact range of a number type using the [`std::numeric_limits`](https://en.cppreference.com/w/cpp/types/numeric_limits.html) class template from the header `<limits>`:

In [ ]:
%%cpp
cout << format("Range of int: {{{} ... {}}}\n", 
               numeric_limits<int>::min(),  
               numeric_limits<int>::max());

The number of values of type `int` can alternatively be computed as follows:

In [ ]:
%%cpp
1uLL + numeric_limits<int>::max() - numeric_limits<int>::min()

You might have noticed that the actual size of `int` is not the same as the required size specified in the C++ standard. Let's ask AI about it:

In [ ]:
%%hermes
Why int in C++ is 32 bits instead of 16 bits required by the C++ standard?

::::{admonition} Hermes
:class: dropdown

First, a small correction to the question: **the C++ standard never requires `int` to be 16 bits — it requires it to be *at least* 16 bits.** It also only requires an *ordering*, not exact widths: `sizeof(short)` ≤ `sizeof(int)` ≤ `sizeof(long)`, with `short` at least 16 bits, `int` at least 16 bits, and `long` at least 32 bits. Within those floors, **the width of `int` is implementation-defined** — the compiler picks it for the target machine. The cell above prints this machine's actual widths (16/32/64/64 here) and checks the ordering.

So `int` is 32 bits on this machine for three reinforcing reasons:

1. **`int` is meant to be the *native* word size.** The whole design is that `int` is "the integer the hardware handles best." On a 32-bit CPU (the x86 era) the natural word is 32 bits, so `int` is 32. On today's 64-bit machines the hardware word is 64, but the `int`/`long`/pointer layout is frozen by the **data model** of the platform, not by the language:
   - **ILP32** (32-bit systems): `int` = `long` = pointers = 32 bits.
   - **LP64** (Linux/macOS x86-64, ARM64): `long` and pointers = 64, but `int` stays **32**.
   - **LLP64** (Windows x86-64): `long long` and pointers = 64, `long` stays 32, `int` stays **32**.
   In every modern model `int` is 32; that's why you see 32 even on 64-bit machines.

2. **Backward compatibility.** If `int` suddenly doubled to 64 bits, every loop counter, array index, and binary layout written for the 32-bit world would change size, break the ABI, and change how much memory programs use. Keeping `int` at 32 lets decades of code run unchanged — which is exactly the portability C and C++ were built to give.

3. **The minimum exists for tiny machines.** The "at least 16" floor is what lets the same source compile on 16-bit microcontrollers (where `int` *is* 16) and on big machines. On such embedded targets, `int` legitimately *is* 16 bits — the standard's floor is met, nothing more. That's precisely why the next cell recommends **fixed-width types** (`int16_t`, `int32_t`, `int64_t` from `<cstdint>`) when you actually need a guaranteed size across platforms.

In short: 32 bits is not something the standard demands — it's the value the implementation chose because 32 is the native, historically-stable word size for `int` on every common platform, and it happens to sit comfortably above the standard's 16-bit floor.

Design notes:
- **The ordering is the real guarantee.** `short <= int <= long` is what the standard actually pins down; the 16/32/64 numbers are *lower bounds*, and on real systems the types are typically exactly 16 / 32 / 32-or-64.
- **Don't write "32" where it matters** — if code depends on the width, use `int32_t` (and friends); if it just needs "a normal integer," `int` is the right, idiomatic choice.

::::

Similarly, the size of `long` can be 32 bits or 64 bits depending on the computer/compiler used. To ensure a definite range across different implementations, fixed integer types are available from [`<cstdint>`](https://en.cppreference.com/w/cpp/types/integer.html).

In [ ]:
%%cpp
1uL - numeric_limits<int16_t>::min() + numeric_limits<int16_t>::max() == 1uL << 16

For instance, [](#code_fast_inv_sqrt1_guess) uses `int64_t` to ensure that the integer type has 64 bits, matching the size of the type of `x` to be explained in the section [](#Floating-Point-Number).

::::{exercise}
:label: ex:limits

Explain where the result is the same if we use `1` is used instead of `1uLL` in the calculation below:

```cpp
1uLL + numeric_limits<int>::max() - numeric_limits<int>::min()
```
    
::::

YOUR ANSWER HERE

### Variable

Knowing how to specify an integer, we need a way to store it and retrieve it to perform more complicated computations.

::::{caution}

C++ is [statically typed](https://en.wikipedia.org/wiki/Type_system#Static_typing), requiring explicit declaration of variable types and making programmers aware of their memory requirements. When done properly, C++ programs can be very fast. Unfortunately, writing good C++ programs can be quite demanding. Mistakes can lead to security risks that are hard to detect or fix.

::::

The following declares integer variables with the default [initialization](https://en.cppreference.com/w/cpp/language/initialization.html) of value `0` for variables with [static storage duration](https://en.cppreference.com/w/cpp/language/storage_duration.html#Static_storage_duration) such as [global variables](https://en.cppreference.com/w/cpp/language/scope.html).

In [ ]:
%%cpp
int a;
short b, c;
unsigned long d, e, f;
cout << format("{} {} {} {} {} {}\n", a, b, c, d, e, f);

`int a;` is an [expression statement](https://en.cppreference.com/w/cpp/language/statements.html#Expression_statements) (ended with a semi-colon) that declares an integer variable named `a`.

A variable name must be a valid [identifier](https://en.cppreference.com/w/cpp/language/identifiers.html). In particular, similar to Python, it should start with a letter or an underscore, and should not be one of the [keywords](https://en.cppreference.com/w/cpp/keywords.html). For instance, the following declarations fail:

```cpp
int 1a;
int this;
```

::::{seealso} Style guides for C++

Similar to [Python's PEP 8](https://peps.python.org/pep-0008/#naming-conventions), C++ also have style guides for naming variables and writing clean code:

- [Google C++ Style Guide](https://google.github.io/styleguide/cppguide.html#General_Naming_Rules)
- [LLVM Coding Standards](https://llvm.org/docs/CodingStandards.html#name-types-functions-variables-and-enumerators-properly)
- [C++ Core Guidelines](https://isocpp.github.io/CppCoreGuidelines/CppCoreGuidelines#main)

::::

Variables without static storage duration such as the following variables defined in a [block scope](https://en.cppreference.com/w/cpp/language/scope.html#Block_scope) are not initialized:

In [ ]:
%%cpp
{
    int a;
    short b, c;
    unsigned long d, e, f;
    cout << format("{} {} {} {} {} {}\n", a, b, c, d, e, f);
}

To initialize the variables explicitly, the following uses the [*copy initialization*](https://en.cppreference.com/w/cpp/language/copy_initialization.html):

In [ ]:
%%cpp
auto a = 1;
short b = a+1, c = b+1;
auto d = 1uL;
cout << format("{} {} {} {}\n", a, b, c, d);

In the above code, `auto` is a [placeholder type specifier](https://en.cppreference.com/w/cpp/language/auto) that allows the compiler to automatically deduce the data type based on the initializer:

In [ ]:
%%cpp
a

In [ ]:
%%cpp
d

Note that a variable is available immediately after the [*locus/point* of declaration](https://en.cppreference.com/w/cpp/language/scope.html#Point_of_declaration). It is really **immediately**:[^recursion]

[^recursion]: This will be useful when defining a [recursion](https://en.wikipedia.org/wiki/Recursion) using [lambda expression](https://en.cppreference.com/w/cpp/language/lambda.html).

In [ ]:
%%cpp
int one=one+1; // `auto` would not work obviously.
one

In [ ]:
%%hermes
Explain briefly whether `one` in the following code is initialized as `one+1`? 
Or `1`? or Or `0`?
---
int one=one+1;

::::{admonition} Hermes
:class: dropdown

:::{caution}

This is an example of AI [hallucination](https://en.wikipedia.org/wiki/Hallucination). The explanations below are not entirely correct. It is important to read the official documentation and verify the behavior by running the code.

:::


**None of the three, guaranteed.** `int one = one + 1;` is **undefined behavior** — the `one` on the right-hand side is the *same* variable being declared, and it has no value yet when the initializer runs. You're reading it before it's initialized.

Why none of `one+1`, `1`, or `0` is a correct answer:
- **Not `one+1`** — there is no earlier `one` to add to; the name doesn't refer to any existing object here.
- **Not `0` by rule** — it's not "zero-initialized" like a static; an automatic variable with no initializer just holds *indeterminate* (garbage) bytes.
- **Not `1` by rule either** — on this machine, in this cell, the garbage happened to be `0`, so `0 + 1` printed `1`. That's luck, not the language.

The cell above makes this concrete: `g++` warns `‘one’ is used uninitialized` (`-Wuninitialized`), and the printed value depends on whatever was already in that stack slot. A different compiler, optimization level, or a few instructions earlier could print anything — `1`, `-1`, a huge number, etc. The *only* thing you can say is that the read is **undefined behavior**, so any value (or a crash) is "correct."

That's also why the note says **`auto` would not work**: `auto one = one + 1;` can't deduce a type from an initializer that itself refers to the very variable being deduced.

Design notes:
- **Point of declaration** — a variable is only usable from *after* its full declaration is done; the initializer still belongs to the declaration, so `one` isn't in scope *with a value* until the initializer is finished. Reading it there is illegal.
- **The sibling cell** `int one(one+1);` is the same trap written in function-style (direct) initialization — also undefined behavior, also flagged by `-Wuninitialized`.

::::

The copy initialization involves an extra copy step, which can be costly for composite data consisting of many values. The extra copy step can be avoided using [*direct initialization*](https://en.cppreference.com/w/cpp/language/direct_initialization.html):

In [ ]:
%%cpp
int one(one+1);
one

Let's play with the copy and direct initialization some more:

In [ ]:
%%cpp
short b(100000);    // runs without error! Why?
b

In [ ]:
%%cpp
unsigned short d=-1;  // runs without warning! Why?
d

::::{caution}

Both the copy and direct initializations do not prevent *narrowing*, i.e., the process of converting a value from a larger data type to a smaller one, potentially resulting in the loss of information or precision.

::::

To catch bugs caused by narrowing, C++11 introduced [*list initialization*](https://en.cppreference.com/w/cpp/language/list_initialization.html):

In [ ]:
%%cpp
int one{one+1};
one

This initialization not only avoid an extra copy step associated with the copy initialization, it also prevents information loss due to narrowing. In particular, the following fails as narrowing is checked by list initialization:

In [ ]:
%%cpptutor
int main() {
    short b{100000};
    unsigned long d{-1};
    return 0;
}

### Operator

The values of variables can be modified using [assignment operators](https://en.cppreference.com/w/cpp/language/operator_assignment.html):

In [ ]:
%%cpp
unsigned long d, e, f;
d = e = f = 1;
cout << format("{} {} {}\n", d, e, f);

`d = e = f = 1` behaves like chained assignment in Python:

In [ ]:
d = e = f = 1
print(d, e, f)

Similar to the augmented assignment operators in Python, C++ also has [compound assignment operators](https://en.cppreference.com/w/cpp/language/operator_assignment.html#Built-in_compound_assignment_operator) such as `+=`, `-=`, `*=`, `/=`, `%=`, `&=`, `|=`, `^=`, `<<=`, `>>=`:

In [ ]:
%%cpp
f += e += d += 1;
cout << format("{} {} {}\n", d, e, f);

However, `f += e += d += 1` is not a valid syntax in Python. How does assignment work in C++?

1. In C++, assignment operators are right associative and so the evaluation is equivalent to
    ```cpp
    f += (e += (d += 1))
    ```
    
    In comparison, assignment operators are non-associative in Python.

2. In C++, an assignment operation has a value equal to the assigned value, so 
    - `(d += 1)` evaluates to `d+1`;
    - `(e += d+1)` evaluates to `e+d+1`; and
    - `(f += e+d+1)` evaluates to `f+e+d+1`.
   
   In comparison, an assignment is usually a statement in Python that does not have a value.[^walrus]

[^walrus]: The exception is the assignment expression using the walrus operator `:=`.

Note that assignment and initialization are different operations. For instance, a variable can be declared to be a [constant](https://en.cppreference.com/w/cpp/language/cv.html) using `const`:

In [ ]:
%%cpp
const int one=one+1; // cannot write `const int one+=1;`
// one=one+1;        // can initialize but not assign a value to a constant
cout << one;

A constant is stored in read-only memory that cannot be modified after initialization. While the default initialization and copy initialization in `const int one=one+1;` are okay, the assignment operation `one=one+1` is not. There is no compound initialization like `const int one+=1;` either.

::::{caution} Precedence and Associativity

An expression often involves many operators, so it is important to learn the precedence and associativity of [the list of operators](https://en.cppreference.com/w/cpp/language/operator_precedence.html) to understand the code. If you get to write the code instead, you can always use parentheses to specify the desired order.

::::

Different programming languages may implement different operators, or have different implementations for the same operator.

For instance, while C++ has no exponentiation operator `**`, unlike Python, it has the suffix/prefix increment/decrement operators `++`/`--`, which are not available in Python:

In [ ]:
%%cpp
int x = 0;
int y = x++; // increments x after evaluation
int z = --x; // decrements x before evalution
x == y && y == z && z == 0  // all zero?

The last expression utilizes the comparison operator `==` (not `=`) to check *equality* and combine these checks using the logical *AND* operator `&&`. In general, comparison operators have higher precedence than logical operators and so they are evaluated first, i.e.,

```cpp
(x == y) && (y == z) && (z == 0)
```

In Python, you can achieve the same effect using a chained comparison:

In [ ]:
ROOT.x == ROOT.y == ROOT.z == 0

::::{exercise}
:label: ex:all-zero

Why does the following C++ code return false even when `x`, `y`, and `z` are all zeros?

::::

In [ ]:
%%cpp
int x = y = z = 0;
x == y == z

YOUR ANSWER HERE

[](#code_gcd1) uses the [modulo operation](https://en.wikipedia.org/wiki/Modulo) `a % b` to give the remainder of `a` divided by `b`. However, the behavior can be confusing for negative operands, i.e., when `a` or `b` are negative. [](#code_gcd1) can give different results if written in a different programming language or hardware because the modulo operator may have a different implementation.

For C++:

In [ ]:
%%cpp
int r0=  5 %  3;
int r1= -5 %  3;
int r2=  5 % -3;
int r3= -5 % -3;
cout << format("{}, {}, {}, {}\n", r0, r1, r2, r3);

For Python:

In [ ]:
print("{}, {}, {}, {}\n".format(5 % 3, -5 % 3, 5 % -3, -5 % -3))

::::{prf:definition} Modulo
:label: def:modulo

$a \bmod b$ gives the remainder $r$ that satisfies for some integer $q \in \mathbb{Z}$, called quotient, that

$$
\begin{align}
a &= b\cdot q + r\\
\lvert r\rvert&<\lvert b\rvert.
\end{align}
$$ (eq:modulo)

The condition above does *not* determine $r$ (and $q$) uniquely unless the sign of $r$ is also specified.

::::

::::{exercise}
:label: ex:modulo

How is the modulo operation implemented differently in C++ and Python?

::::

YOUR ANSWER HERE

The quotient $q$ in [](#eq:modulo) can be obtained by the division operation `/` operation.

In [ ]:
%%cpp
int q0=  5 /  3;
int q1= -5 /  3;
int q2=  5 / -3;
int q3= -5 / -3;
cout << format("{}, {}, {}, {}\n", q0, q1, q2, q3);

In Python, `//` is used instead for integer division. Similar to `%`, the implementation for `//` is also slightly different from `/` in C++.

In [ ]:
print("{}, {}, {}, {}\n".format(5 // 3, -5 // 3, 5 // -3, -5 // -3))

In [ ]:
%%hermes
Explain why integer division is implemented differently in 
C++ versus Python.

::::{admonition} Hermes
:class: dropdown

There is no single "correct" rule for what $a/b$ should return when the result is not an integer — you must **drop part of the fractional part, and there are two natural ways to do it.** C++ and Python simply chose different ones, for different reasons.

**Both languages agree on the identity** $a = q\cdot b + r$ with $|r| < |b|$. What differs is *which integer* $q$ lands on, and the remainder $r$ is forced to follow:

- **C++ truncates toward $0$** (round $a/b$ down in *magnitude*). The remainder $r = a - qb$ then carries the **sign of the dividend** $a$ — so $-5/3 = -1$ and $-5 \% 3 = -2$.
- **Python floors toward $-\infty$** (round $a/b$ *down* to the next smaller integer). The remainder then carries the **sign of the divisor** $b$ — so $-5//3 = -2$ and $-5 \% 3 = 1$.

**Why they differ — the design motivation:**

- **C++ reflects the hardware.** On the machines C++ targets, integer division is a single `idiv` instruction that *truncates toward zero* (it keeps the top word of the quotient). The language exposes that behavior directly, so `a/b` and `a%b` share the dividend's sign and the two operations stay consistent with one instruction.
- **Python prioritises mathematical and modular convenience.** Floor division makes the remainder always **non-negative when the divisor is positive** ($0 \le a \% b < b$), which is exactly the textbook definition of a residue and what you want for indexing, wrapping, and modular arithmetic — e.g. $(-5) \% 3 = 1$ is the same residue class as $1$, so $a \% b$ is ready to use as a $0$-based index without a sign fix-up.

In short: **C++ optimises for the machine (truncate, match `idiv`); Python optimises for the maths (floor, non-negative residues).** The code cell above reproduces both rules with the same two-line identity and shows the results line up with the lecture's `5/3, -5/3, 5/-3, -5/-3` examples.

Design notes:
- `div_py` is implemented *on top of* the C++ divide: take the truncating quotient, then step down once more if the remainder's sign disagrees with the divisor's — the floor is at most one unit away from the truncation.
- `a == q*b + r` is printed for both and is always `1` (true): both schemes are valid Euclidean divisions, they just split the quotient/remainder differently.

::::

## Character

How to represent a character?

A character literal is a character delimited by *single* quotes.

In [ ]:
%%cpp
'f'

Each value of type [`char`](https://en.cppreference.com/w/cpp/language/types.html#Character_types) is represented by 1 byte.

In [ ]:
%%cpp
sizeof(char)

[`char`](https://en.cppreference.com/w/cpp/language/types.html#Character_types) is actually an integer type, e.g., we can initialize variables of type `char` with integer values as follows.

In [ ]:
%%cpp
char a = 65, b(66), c {67};
cout << format("{} {} {}", a, b, c);

We can also perform arithmetic operations on characters like what we can do on integers:

In [ ]:
%%cpp
a - b * 2

`char` can also be signed or unsigned:

In [ ]:
%%cpp
numeric_limits<char>::min()           // sign bit (left-most bit) equals 1

In [ ]:
%%cpp
numeric_limits<unsigned char>::min()  // 0

A character is represented by an integer according to the [ASCII code](https://en.cppreference.com/w/cpp/language/ascii.html). The following converts between `int` and `char` using [static type casting](https://en.cppreference.com/w/cpp/language/static_cast.html):

In [ ]:
%%cpp
cout << format("The ASCII code of {} is {}.\n", static_cast<char>(65), static_cast<int>('A'));

::::{caution} C-style cast

The [C-style cast](https://en.cppreference.com/w/c/language/cast.html) `(int) 'A'` also works, but it is not preferred because it lacks compile-time type safety checks.
For example, [](#code_fast_inv_sqrt1_guess) for computing the initial guess of the [fast inverse square root algorithm](https://en.wikipedia.org/wiki/Fast_inverse_square_root#Overview_of_the_code) can use the C-style cast like

```cpp
auto i = *(int64_t *)(&x);
...
auto y = *(double *)(&i);
```

The first line converts the address `&x` of a variable `x` of type `double`[^float_later] to an address of `int64_t`, so the dereferenced value can be assigned as an integer to `i`.[^address_later] In other words, while the binary sequences stored in the memory locations of `x` and `i` are the same, they evaluates to different numbers of different types. 

Such a pointer conversion is rare and is often a result of mistakenly typing `*`. Therefore, `static_cast` incorporates type safety checks to raise an error for [such pointer conversions](https://en.cppreference.com/w/cpp/language/static_cast.html) that are not commonly used, helping to identify issues more easily. If such a conversion is intended, however, programmers can use [`reinterpret_cast`](https://en.cppreference.com/w/cpp/language/reinterpret_cast.html) instead, as shown in [this code](#code_fast_inv_sqrt1_guess).

::::

[^float_later]: The type `double` will be explained in [a later section on floating point number](#Floating-Point-Number).
[^address_later]: The address-of and dereference operators will be explained in [a later section on variable scope](#Scope).

In [ ]:
%%hermes
List and explain very briefly the different types of cast in C++. Give
a one-liner example for each cast.

::::{admonition} Hermes
:class: dropdown

C++11 replaced the C-style cast with four named casts, each doing *one* thing with a compile-time check. The cell above exercises all of them (the lines are its real `g++` output, not a sample).

- **`static_cast`** — the workhorse: *reasonable* value conversions (numeric, or up/down casts in a class hierarchy). `int n = static_cast<int>('A');` → `65`.
- **`const_cast`** — the only legal way to add or remove `const`/`volatile`. `int* pc = const_cast<int*>(&cv);` (forming the pointer is fine; writing through it is undefined behavior).
- **`reinterpret_cast`** — low-level re-interpretation between unrelated types, or pointer ↔ integer. *Same bytes, new type.* `int64_t bits = reinterpret_cast<int64_t&>(dd);` is the bit-trick behind the [fast inverse square root](#code_fast_inv_sqrt1_guess) from earlier.
- **`dynamic_cast`** — the safe down/cross-cast in a *polymorphic* hierarchy. It checks at runtime and returns `nullptr` on failure (throws `bad_cast` for references). `dynamic_cast<D*>(&plain)` → `nullptr` because a `B` is not really a `D`.
- **C-style cast** `(int)'A'` — the legacy *kitchen sink*: it silently picks whichever of the four above fits, with none of the type-safety checks. That is why the lecture prefers the named casts; a `(T*)` pointer-cast that `static_cast` would reject quietly becomes a `reinterpret_cast` in disguise.

The one-line mapping: **`static_cast`** = value/type you *mean*; **`const_cast`** = drop `const`; **`reinterpret_cast`** = same bits, new type; **`dynamic_cast`** = is this really a subclass?; **`(T)x`** = all of the above, unchecked.

::::

Be careful that not every character can be printed. E.g., the last character in the ASCII code is <kbd>DEL</kbd> (delete), which cannot be printed.

In [ ]:
%%cpp
cout << format("The ASCII code of {} is {}.\n", static_cast<char>(127), 127);

::::{exercise}
:label: ex:ascii

Explain why printing `static_cast<char>(128), 128)` fails with an error?

::::

In [ ]:
%%cpp
// cout << static_cast<char>(128);   // fails
static_cast<char>(128)               // works

YOUR ANSWER HERE

As another example, the first character in the ASCII code is <kbd>NUL</kbd> (null), which cannot be printed either.

In [ ]:
%%cpp
cout << format("The ASCII code of {} is {}.\n", static_cast<char>(0), 0);

Note that it is missing a few more characters at the end, namely, `is 0.`. Why?

In [ ]:
%%hermes
Why the following C++ code only prints "The ASCII code of"?
---
cout << format("The ASCII code of \{\} is \{\}.\n", static_cast<char>(0), 0);

::::{admonition} Hermes
:class: dropdown

Nothing is truncated by the C++ program — it writes all 26 bytes of the formatted string. The display is cut short because the string **contains a NUL byte**.

The cell above is executed in the cell just above — its hex dump is its real output, not a sample. Formatting `static_cast<char>(0)` inserts the NUL *character* (byte value `00`) into the string, right after "The ASCII code of ":

```
The formatted string has 26 bytes:
54 68 65 20 41 53 43 49 49 20 63 6f 64 65 20 6f 66 20 00 20 69 73 20 30 2e 0a
```

That `00` at position 18 is the embedded NUL. The terminal / notebook display treats a NUL byte like the end of the text (it is how C-style strings are terminated), so the visible output stops after `The ASCII code of` — even though `is 0.` (bytes `69 73 20 30 2e 0a`) was actually written to stdout after it. The same thing happens in the preceding lecture cell; it is not a bug in `format`.

Design notes:
- `char` is an integer type, so `format("{}", c)` accepts it and prints the character it denotes — including control characters such as NUL. For the *value*, format the character *as an integer*: `(int)c0` (or `static_cast<int>(c)`).
- The string object itself is complete (26 bytes here); only its *presentation* is cut at the NUL. This is why `static_cast<char>(128)` "works" when used as an expression value (cell `8df9631f`) but printing it behaves differently — the value is fine, displaying it is the issue.

::::

## Floating Point Number

### Declaration

Computations on real numbers are essential for many applications such as simulations, modeling, computer graphics, and machine learning, etc. To manipulate real numbers, a simple idea is to approximate a real number by a rational number, which can then be represented by two integers, namely, the numerator and denominator. For instance, 

$$\pi\approx \frac{22}{7}.$$

In [ ]:
%%cpp
22/7

That is not quite what we expected! While you may know how to fix the above issue, it highlights the need for a more convenient representation for numerical computations—[floating-point arithmetics](https://en.wikipedia.org/wiki/IEEE_754).[^FPU] 

[^FPU]: Modern CPUs and GPUs are equipped with dedicated hardware called [Floating Point Units (FPUs)](https://en.wikipedia.org/wiki/Floating-point_unit) specifically designed to handle floating-point arithmetic efficiently.

In [ ]:
%%cpp
22/7.  // the point is not the period

::::{exercise}
:label: ex:op_overloading

Explain why `22/7` and `22/7.` produce different results.

::::

YOUR ANSWER HERE

In C++, there are two floating-point data types: `double` and `float`.

In [ ]:
%%cpp
constexpr auto PI = 0.314e1  // scientific notation

The qualifier [`constexpr`](https://en.cppreference.com/w/cpp/language/constexpr.html) declares a constant expression whose value is known at compile time and will not be modified later. In comparison, the qualifier `const` can declare a runtime constant whose value need not be determined at compile time.[^constexpr] For instance, a number randomly drawn with `std::rand()` from `<cstdlib>` can be defined as a constant, but not a constant expression:

[^constexpr]: Constant expressions potentially allows the compiler to further optimize the code for faster execution.

In [ ]:
%%cpp
const auto a=rand();
a

The following will fail because the initializer is randomly drawn at runtime:

In [ ]:
%%cpptutor
int main() {
    constexpr auto a=rand();
    return 0;
}

To enter a `float`, add the suffix `f` to the floating point number:

In [ ]:
%%cpp
constexpr auto PI = 3.14f  // single precision instead of the default double precision

::::{exercise}
:label: ex:float_suffix

The following fails to define `x` as `0` of type `float`. Why?

::::

In [ ]:
%%cpptutor
int main() {
    auto x = 0f;
    return 0;
}

YOUR ANSWER HERE

### Precision

The benefit of `float` is that it occupies less memory than `double`:

In [ ]:
%%cpp
sizeof(double)

In [ ]:
%%cpp
sizeof(float)

However, the smaller memory footprint comes at the cost of a lower precision. For instance, consider the [mass-energy equivalence](https://en.wikipedia.org/wiki/Mass%E2%80%93energy_equivalence):

In [ ]:
%%cpp
constexpr float c = 2.99792458e8f;  // the speed of light
float m = 5.0f, E = m*c*c;        // the mass-energy equivalence

Of course, $E=mc^2$ as verified below:

In [ ]:
%%cpp
(E == m*c*c)

But $\frac{E}{c^2} \neq m$ somehow:

In [ ]:
%%cpp
E/(c*c) == m

Changing the order of operations seems to fix it:

In [ ]:
%%cpp
E/c/c == m

Using `double` instead of `float` also works:

In [ ]:
%%cpp
constexpr double c = 2.99792458e8;
double m = 5, E = m*c*c;
E == m*c*c && E/(c*c) == m

You might think that the issue has to do with very large/small numbers. The following shows that even a number close to 1, and with just one decimal place cannot be accurately represented in floating point:

In [ ]:
%%cpp
cout << fixed << setprecision(20) << 1.1 << '\n';  // for double

::::{seealso} How does the above code print up to 20 decimal places?
:class: dropdown

- [`std::fixed`](https://en.cppreference.com/w/cpp/io/manip/fixed.html): This manipulator sets the output format to fixed-point notation.
- [`std::setprecision(20)`](https://en.cppreference.com/w/cpp/io/manip/setprecision.html) from [`<iomanip>`](https://en.cppreference.com/w/cpp/header/iomanip.html): This sets the number of digits after the decimal point to 20.

::::

Similarly for Python:

In [ ]:
print(f"{1.1:.20f}")

Similarly for `float`:

In [ ]:
%%cpp
cout << format("{:.20f}\n", 1.1f);  // for float

::::{seealso} How does the above code format the floating point number?
:class: dropdown

The code uses a [format specifier](https://en.cppreference.com/w/cpp/utility/format/spec.html) `{:.20f}`, which is a little bit cryptic but very convenient.

::::

Floating-point numbers have *limited precision*:

- Single precision is accurate typically to 6-9 decimal digits.
- Double precision is accurate typically to 15-17 decimal digits.

The limits of `float` (and similarly `double`) can be obtained as follows from [`numeric_limits`](https://en.cppreference.com/w/cpp/types/numeric_limits.html#Member_functions):

In [ ]:
%%cpp
cout << format("Minimum value: {}\n", std::numeric_limits<float>::min());
cout << format("Lowest value (including subnormal): {}\n", std::numeric_limits<float>::lowest());
cout << format("Maximum value: {}\n", std::numeric_limits<float>::max());
cout << format("Epsilon (difference between 1.0 and the next representable float): {}\n", std::numeric_limits<float>::epsilon());
cout << format("Round error: {}\n", std::numeric_limits<float>::round_error());
cout << format("Denormalized minimum value: {}\n", std::numeric_limits<float>::denorm_min());

In [ ]:
%%cpp
numeric_limits<float>::min()           // the smallest positive normal value

In [ ]:
%%cpp
numeric_limits<float>::lowest()        // the lowest finite value

In [ ]:
%%cpp
numeric_limits<float>::max()           // the largest finite value

In [ ]:
%%cpp
numeric_limits<float>::epsilon()       // the gap from 1.0 to the next value

In [ ]:
%%cpp
numeric_limits<float>::round_error()   // the maximum rounding error

There are also some special values defined according to the IEEE 754 standard:

In [ ]:
%%cpp
numeric_limits<float>::infinity()      // the positive infinity value

In [ ]:
%%cpp
numeric_limits<float>::quiet_NaN()     // a quiet NaN value

In [ ]:
%%cpp
numeric_limits<float>::signaling_NaN() // a signaling NaN value

In [ ]:
%%cpp
numeric_limits<float>::denorm_min()   // the smallest positive subnormal value

Python's `float` is different from C++'s `float` because it has double precision:

In [ ]:
import sys
sys.float_info

To understand the precision issue of floating point numbers, play with a simulator:

- [IEEE 754 Floating Point Converter](https://www.h-schmidt.net/FloatConverter/IEEE754.html)
- [Float Toy](https://evanw.github.io/float-toy/)

The following is an IEEE 754 simulator written in Python:

In [ ]:
@interact(x=FloatLogSlider(
    value=1,        # Initial value of the slider
    base=2,         # Base of the logarithm (e.g., 10 for base-10 log)
    min=-1023-52,
    max=1023,
    step=1,
    description='x' # Label for the slider
))
def double2binary(x):
    # Convert the double to its binary representation
    binary = f"{unpack('>Q', pack('>d', x))[0]:064b}"
    
    # Extract sign, exponent, and mantissa
    sign = binary[0]
    exponent = binary[1:12]
    mantissa = binary[12:]
    
    # Convert exponent and mantissa to decimal
    sign_val = int(sign, 2)
    exponent_val = int(exponent, 2)
    mantissa_val = int(mantissa, 2)/2**52
    
    # Create color-coded HTML output 
    html_output = (
        f"Binary: "
        f"<span style='color:red;'>{sign}</span>"
        f"<span style='color:green;'>{exponent}</span>"
        f"<span style='color:blue;'>{mantissa}</span><br>"
    ) 
    html_output += (
        f"$(-1)^{{\\color{{red}}{sign_val}}}\\times "
        f"2^{{{{\\color{{green}}{exponent_val}}}-1023}}\\times "
        f"(1+{{\\color{{blue}}\\text{{{mantissa_val}}}}})$"
    ) if exponent_val < 2047 else (
        r"NaN" if mantissa_val > 0 else (
            f"${('', '-')[sign_val]} \\infty$"
        )
    )

    display(HTML(html_output))

In [ ]:
double2binary(float('inf'))

In [ ]:
double2binary(-float('inf'))

In [ ]:
double2binary(float('nan'))

::::{caution} Why does the precision error depend on the order of operations?
:class: dropdown

Floating-point numbers operate at varying scales due to their exponents, so using the same number of bits for the mantissa can lead to precision errors that differ by scale. The order of operations affects how these errors accumulate, resulting in varying overall precision depending on the scale.

::::

::::{exercise}
:label: ex:max_double

Explain why the followings are true?

::::

In [ ]:
%%cpp
const double m=1e16;
m - 1 == m

In [ ]:
%%cpp
const double m=1e100;
m*m*m*m == m*m*m*m*m*m*m*m*m*m*m*m*m*m*m*m*m*m*m*m

YOUR ANSWER HERE

::::{exercise}
:label: ex:nan

Explain why the mass of an atom is not equal to itself.

::::

In [ ]:
%%cpp
constexpr float mass_of_universe = 1.45e53;
constexpr float num_of_atoms = 1e80;
const float mass_of_atom = mass_of_universe/num_of_atoms;
(mass_of_atom == mass_of_atom)

YOUR ANSWER HERE

::::{note}

`mass_of_atom` is declared with the [`const` type qualifier](https://en.cppreference.com/w/c/language/const.html) instead of `constexpr` because its value is not known at compile time even if it is expected to be a constant.

::::

## Scope

The access of a variable is restricted to its [scope](https://en.cppreference.com/w/cpp/language/scope.html). For instance, in [](#code_gcd1), `a` and `b` can be accessed anywhere since they have global scope, but `c` can only be accessed within the [compound statement](https://en.cppreference.com/w/cpp/language/statements.html#Compound_statements) enclosed by the braces `{ ... }`, which creates a [block scope](https://en.cppreference.com/w/cpp/language/scope.html#Block_scope).

```cpp
{
  ...
  int c = b;
  ... // c visible here
} // c is out of scope
```

::::{caution} Redeclarations of a variable

You might wonder why we use a compound statement. This is because, even though the `cling` interpreter allows redeclarations of a variable in separate runs, C++ compilers do not allow redeclaring a variable within the same scope. The code inside the block needs to be executed repeatedly to produce the final result.

::::

C++ follows [lexical scoping](https://en.wikipedia.org/wiki/Scope_(computer_science)) to access variables defined in the closest enclosing scope. To understand how this works, consider the following example:

In [ ]:
%%cpp
// global scope
int a;
{ // block scope level 1
    { // block scope level 2
        int a;
        { // block scope level 3
            cout << "Level 3: a=" << a << '\n';
        }       
    }
    cout << "Level 1: a=" << a << '\n';
}

- The first `cout` in level 3 accesses `a` defined in level 2, which *shadows* the variable `a` in the global scope. Note that local variables are not initialized to `0` by default.
- The second `cout` in level 1 accesses `a` defined in the global scope, which is initialized to `0` by default. `a` defined in level 2 is out of the scope of level 1.

In C++, a variable is not merely a name; it is a named container whose size is determined by its type. In comparison, Python is [dynamically typed](https://en.wikipedia.org/wiki/Type_system#DYNAMIC). Instead of a memory location, a variable in Python can be considered simply as a name of an object. In particular, the memory locations of different variables can be the same:

In [ ]:
a = b = 1
print(f"a={a} @ {id(a):#x}")
print(f"b={b} @ {id(b):#x}")

The above uses `id` in CPython, which returns the memory location of its argument. The assignments above are called aliasing, since both `a` and `b` are different names pointing to the same memory location.

::::{tip}

To learn more about a Python function, we can use the contextual help by placing the cursor over a function name and 
- click the menu item `Help`$\to$`Show Contextual Help` or
- press the short-cut key <kbd>Shift + Tab</kbd>.

::::

For C++, different variables *normally* have different memory locations even if they have the same value.

In [ ]:
%%cpp
int a=1, b=a;
cout << format("a={} @ ", a) << &a << '\n';
cout << format("b={} @ ", b) << &b << '\n';

The above code uses the [address-of operator `&`](https://en.cppreference.com/w/cpp/language/operator_member_access.html#Built-in_address-of_operator), which returns the address of type `int *`:

In [ ]:
%%cpp
&a

Variables with the same name also have different memory locations:

In [ ]:
%%cpp
int a=1;
{
    int a=++a;
    cout << format("a={} @ {:p}\n", a, static_cast<void*>(&a));
}
cout << format("a={} @ {:p}\n", a, static_cast<void*>(&a));

::::{note}

The above code uses `static_cast<void*>` to convert `&a` to type `void*` so it can be formatted as an address with the format specifier `{:p}`.

::::

::::{caution} Shouldn't the code prints `a=2 @ ...` first?
:class: dropdown

`++a` actually uses the already declared `a` in the block scope, whose value is uninitialized and therefore may not be `1`.

::::

We can store the address using a variable known as a [pointer](https://en.cppreference.com/w/cpp/language/pointer.html):

In [ ]:
%%cpp
int* p=&a;
cout << format("a={}\n", *p);

`*p` above uses the [indirection/dereference operator `*`](https://en.cppreference.com/w/cpp/language/operator_member_access.html#Built-in_indirection_operator) to access the value that `p` points to. Indeed, since it is far more common to operate on `*p` instead of `p`, the declaration for multiple pointers requires specifying `*` for each pointer:

In [ ]:
%%cpp
int *p=&a, *q=&b;
cout << format("a={}, b={}\n", *p, *q);

The default initialization for pointers with static storage duration is `nullptr` or `0`, referred to as the null pointer. The value indicates that the pointer does not point to any object, i.e.,  dereferencing it leads to an error:

In [ ]:
%%cpp
int *p, *q=0, *r=nullptr;  // global p is initialized to nullptr by default
p==q && q==r               // nullptr has an integer value 0

::::{caution}

Using an uninitialized pointer is unsafe. For instance:

```cpp
{
    int *p;
    cout << *p;  // 👨🏻‍🏫 ❌ Undefined behavior
    *p = 1;      // 👨🏻‍🏫 ❌ Dangerous: p points to an arbitrary memory location
}

```

::::

In [ ]:
%%cpp
{
    int *p;
    cout << *p;
    // *p = 1; // 😈: If you never try, you'll never know.
}

The use of pointers can get rather complicated as can be seen in [](#code_fast_inv_sqrt1_guess). The following code uses the idea to give an IEEE 754 simulator in C++:

In [ ]:
%%cpp
double x=1.;
auto i = reinterpret_cast<int64_t *>(&x);
cout << format("{:064b}", *i);

The above prints the binary representation of the integer `i`, which is also the binary representation of the floating point number `x`.

The effect of aliasing can also be achieved using pointers.

In [ ]:
%%cpp
int a=1, *b=&a;
cout << format(" a={} @ {:p}\n", a++, static_cast<void*>(&a));
cout << format("*b={} @ {:p}\n", *b, static_cast<void*>(b));

Aliasing can also be achieved in C++ using an [*lvalue (locator value) reference*](https://en.cppreference.com/w/cpp/language/reference.html) such as `int &`:

In [ ]:
%%cpp
int a=1, &b=a;
cout << format("a={} @ {:p}\n", a++, static_cast<void*>(&a));
cout << format("b={} @ {:p}\n", b, static_cast<void*>(&b));

The increment `a++` also modifies the value of `b` because the declaration `int &b=a;` binds the address of `a` to `b`, so both `a` and `b` share the same memory location. The behavior of aliasing in Python is still very different:

In [ ]:
b = a = 1  # a and b have the same memory location
a = a + 1  # a incremented to 2
b          # b is also 2? a and b have the same memory location right?

::::{caution} Why `b` is not equal to `2`?
:class: dropdown

`b` is still `1` because it is not an alias of `a`, but rather, an alias of the object/integer `1`. 
- `a = a + 1` in Python assign the name `a` to a new value `a + 1`, without changing the value `1` that `b` points to.
- `a++` in C++ increments the value `a`, which is also the value of `b`.

::::

In C++, an integer such as `1` also has an associated memory location, but it is temporary and optimized heavily by the compiler in a way not suitable for users to modify. `int &b=1;` fails because, otherwise, the location of `1` would be exposed to the users to modify. We say that `1` is not an lvalue, but rather, an rvalue, or more specifically, a [prvalue](https://en.cppreference.com/w/cpp/language/value_category.html#prvalue).

It is okay, however, if `b` is declared as a constant:

In [ ]:
%%cpp
const int &b=1

The compiler can safely extend the lifetime of the prvalue as long as `b` is in scope, without concern about modifications or interference with optimizations for temporary objects. Such a constant reference will be useful when passing larger objects around without the extra copy step as in the copy assignment.

In [ ]:
%%hermes
Explain briefly how pass by constant reference works in C++.

::::{admonition} Hermes
:class: dropdown

Pass by constant reference declares a parameter as `const T&` — a *read-only alias* for the caller's argument, not a copy. Why the `const` matters:

- **Read-only inside the function.** The function cannot modify the caller's object, and the contract is visible in the signature.
- **No copy.** For a large object (e.g., `const std::string&`), binding a reference costs a handful of bytes as compared to copying the whole object. Pass by reference avoids the copy, and `const` adds the safety of guaranteeing no modification.
- **Rvalues are admissible.** A `const T&` parameter can bind to temporaries (as `123` did above), which is why library functions that only *read* their inputs declare them this way; a plain `T&` cannot bind to `123` at all.

::::

::::{exercise}
:label: ex:pre_vs_post

Why `++a=1` works but `(a++)=1` fails?

::::

YOUR ANSWER HERE

In [ ]:
%%hermes
Explain the value categories of C++ with concrete examples.

::::{admonition} Hermes
:class: dropdown

C++ gives every *expression* one of three **value categories**: **lvalue**, **prvalue**, or **xvalue**. The fundamental split is lvalue vs. rvalue: an *lvalue* names a region of memory that persists — it has *identity* (you can take its address and use it again) — while an *rvalue* is a value to be consumed. **Rvalue** is the umbrella term for the other two: a **prvalue** is a pure value with no identity of its own, and an **xvalue** is an expiring object (it keeps identity but is explicitly marked movable).

[](./value_categories.ipynb)

**Part A — the lvalue / rvalue split, computed by overload resolution.** `category` is overloaded on `T&` versus `T&&`: an lvalue argument binds `T&` (which is preferred), but an rvalue argument can only bind `T&&`, so the `T&&` overload catches exactly the rvalues. The output confirms the usual rules:

- `demo_a`, `s`, and `s[0]` are **lvalues** — a variable and a subscript both name a region of memory.
- `demo_a + 1`, the temporary `std::string("world")`, and `s.size()` (returned by value) are **rvalues**.
- `std::move(demo_a)` and `std::move(s)` are **rvalues** — `std::move` moves nothing by itself; it only casts to an rvalue reference.

**Part B — rvalues split into prvalue and xvalue.** A single function parameter cannot tell these two apart (both bind `T&&`), so `rvalue_kind` inspects `decltype` instead: an *xvalue* still refers to an object, so `decltype(std::move(demo_a))` is `int&&`, whereas a *prvalue* is a pure value, so `decltype(demo_a + 1)` is `int`. Hence `std::move(demo_a)` is an **xvalue** but `demo_a + 1` and the new string are **prvalues**.

**Part C — the `++` asymmetry the lecture alluded to.** `++c` returns a reference *to* `c`, so it is an lvalue; `d++` returns a *copy*, so it is a prvalue. That is precisely why `(++c) = 1` compiles while `(d++) = 1` does not — the real compiler rejects the latter with `lvalue required as left operand of assignment` (see [](#ex:pre_vs_post)).

**Part D — the payoff: an xvalue enables a move.** Binding the lvalue `p` to a `ValueCatDemo&&` parameter via `std::move(p)` turns it into an xvalue, so the move constructor runs and `p` is drained; copying the lvalue `q` runs the copy constructor instead. Real output:

```
p (prvalue in)  / q (from xvalue) / r (from lvalue):
  move constructor (from n=3)
  copy constructor (from n=3)
p=0 q=3 r=3 (p was emptied by the move)
```

In one line: an **lvalue** is an addressable object, an **xvalue** is an expiring object (the one that can be *moved*), and a **prvalue** is a value with no identity of its own (a literal, a computed result, a by-value return). It is what lets the compiler choose between copying and moving, and lets references bind to temporaries — the same reason a `const int&` was able to bind the literal `123` in the previous cell.

:::{caution}

As AI can [hallucinate](https://en.wikipedia.org/wiki/Hallucination). Verify the code works in a separate notebook running C++23 kernel.

:::

::::

[^hermes_response]: This is a sample response generated by the above prompt. Click me to expand the content.

::::{exercise}
:label: ex:uninit

Since different variables occupy different memory locations, can we modify [](#code_gcd1) as follows to avoid overwriting the original `a` and `b`?

```cpp
{
    int a=a, b=b;
    ...
}
```

Why or why not?

::::

YOUR ANSWER HERE

As a hint, try running the following program to check whether the local variables are actually clones of the global variables:

In [ ]:
%%cpp
int a=2*3*4, b=3*4*5;
{
    int a=a, b=b;
    cout << format("a={}\nb={}\n", a, b);  // local clone of global a and b?
}
cout << format("a={}\nb={}\n", a, b);  // global a and b not overwritten

## String

How to represent a piece of text, which consists of a sequence of characters?

A string literal is delimited by double quotes:

In [ ]:
%%cpp
"15"

Note that the data type is not `string` but `const char[3]`: a constant array of 3 `char` values. Such an array of characters is referred to as a *C string*, even though there isn't actually a distinct `cstring` data type in C.

Why are there 3 characters instead of 2? It’s almost as if there’s a hidden element at play. We can inspect the elements of the array using a [member access operator like `[]`](https://en.cppreference.com/w/cpp/language/operator_member_access.html), but what secrets might it unveil?

In [ ]:
%%cpp
"15"[0] // [0] picks out the first character at index 0

In [ ]:
%%cpp
"15"[1] // the second character

Lo and behold 🪄:

In [ ]:
%%cpp
"15"[2] // the third character

The last character is the *null character*, which can also be entered as `'\0'`.

In [ ]:
%%cpp
'\0'

C strings are *null-terminated* to ensure they are [(uniquely) decodable](https://en.m.wikipedia.org/wiki/Variable-length_code#Uniquely_decodable_codes). In other words, the null character signals the end of a string without needing to keep track of its length. What a clever mechanism!

Can you print a string containing `\0` in the middle?

In [ ]:
%%cpp
cout << "321\0 123";

In [ ]:
%%cpp
cout << "321\0123";

::::{exercise}
:label:ex:0_in_middle

Explain below what gets printed in the above two code cells. 

::::

YOUR ANSWER HERE

::::{exercise}
:label: ex:index_OOB

Why does the following code even run in C++?

:::{hint}
:class: dropdown

The issue is called [buffer overrun](https://en.wikipedia.org/wiki/Buffer_overflow). Try running the same code in python as a comparison.

:::

::::

In [ ]:
%%cpp
"15"[3]

YOUR ANSWER HERE

The length of a C string can be returned using [`std::strlen`](https://en.cppreference.com/w/cpp/string/byte/strlen.html) from [`<cstring>`](https://en.cppreference.com/w/cpp/header/cstring.html):

In [ ]:
%%cpp
strlen("15")

Note that the return type is [`std::size_t`](https://en.cppreference.com/w/cpp/types/size_t.html), which is also defined in other header files such as [`<cstddef>`](https://en.cppreference.com/w/cpp/header/cstddef.html). Failure to understand the implication could lead to logical error like the following:

In [ ]:
%%cpp
strlen("Ava") - strlen("Betty") > 0

In [ ]:
%%hermes
In C++, should `strlen("Ava") - strlen("Betty") > 0` return false since 
"Ava" is shorter than "Betty"?

::::{admonition} Hermes
:class: dropdown

The expression returns `true`, not `false`. This is an unsigned arithmetic trap. Let me read the surrounding cells for context.




Yes — intuitively it *should* be `false` (`"Ava"` is 3 chars, `"Betty"` is 5, so `Ava` is shorter). But in C++ it actually returns **`true`**. That mismatch is exactly the point of this cell (the markdown just above it calls this "a logical error" from misunderstanding the return type).

**Why it's `true`**

`std::strlen` returns `std::size_t`, which is an **unsigned** integer. So the subtraction is done in unsigned arithmetic, and unsigned subtraction wraps around instead of going negative:

```cpp
strlen("Ava") - strlen("Betty")
// 3 - 5 in size_t = 2^64 - 2 = 18446744073709551614
```

That huge positive number is, of course, `> 0`, so the whole expression is `true`.

I verified it by compiling:

```
strlen("Ava") = 3
strlen("Betty") = 5
diff = 18446744073709551614
diff > 0 ? 1            // true  <-- the buggy version
int version: 0          // false <-- if the operands were signed
```

**The general trap**

This is the classic *unsigned wraparound* bug. It shows up everywhere you compare two `size_t` values directly:

```cpp
size_t a = 3, b = 5;
a - b > 0;   // true, but a < b!
```

**How to write it correctly** — compare the lengths directly, and never subtract two unsigned values expecting a negative:

```cpp
strlen("Ava") > strlen("Betty");   // false, correct
// or, to express "Ava is shorter":
strlen("Ava") < strlen("Betty");   // true
```

So your instinct is right about the *intended* semantics — the lesson is that C++ won't give you that signed result automatically; the `size_t` return type silently turns your `a - b > 0` idiom into a tautology. Compare the operands, don't subtract-then-compare.

::::

::::{exercise}
:label: ex:unsigned

What is the maximum possible length for C string?

::::

YOUR ANSWER HERE

In [ ]:
%%hermes
Since the length of a C string is recorded in its type, why do we need to 
null-terminate it? Isn't it a waste of memory to store the null character?

In [ ]:
%%cpp
/** The array type records its size only at compile time. Once the array decays to a
    pointer — as `strlen` receives it — the type no longer records the length, and
    the null terminator is the only record of the length left at runtime. */
{
    const char text[3] = "15"; // array type: 3 chars, null terminator included
    const char* p = text;      // decayed to a pointer: the type no longer records the length

    format("sizeof(text)  = {}   // 3 is part of the array type (compile time)\n", sizeof(text));
    format("sizeof(p)     = {}   // the size of a pointer, not of the array\n", sizeof(p));
    format("strlen(text)  = {}   // scan text[0], text[1], stop at the null character\n", strlen(text));
    format("sizeof(text) == strlen(text) + 1: {}   // the null character is the extra byte\n", sizeof(text) == strlen(text) + 1);
}

::::{admonition} Hermes
:class: dropdown

First, a small correction: the length is recorded in the type **only while the string is still an array**. The literal `"15"` has type `const char[3]`, and that 3 is known at **compile time** only. The moment the array is used as a pointer — which is what every string function actually receives — the type no longer records any length at all, and the null terminator is the only record of the length left at **runtime**. That is why it is needed.

**Is the byte wasted?** It is a constant one-byte overhead, and in exchange C gets a runtime string representation that is just a plain sequence of bytes: no header, no length field to keep in sync, no allocator, and the string can live in read-only program memory (which is exactly where literals do). The alternative design — storing the length *alongside* the characters, a length-prefixed string — is what `std::string` and most other languages' string types use, and it can be more space-efficient. C deliberately traded that for a minimal, headerless representation in which any function can find the end of the string by scanning, with nothing to pass in besides the pointer.

::::

Another frustrating gotcha is that the length of a string is **not** the same as the number of characters or symbols in the string! Consider the length of ["思"](https://translate.google.com/?sl=yue&tl=en&text=%E6%80%9D&op=translate):

In [ ]:
%%cpp
strlen("思")

Why is the length 3 instead of 1?

In [ ]:
%%cpp
"思"[0]

In [ ]:
%%cpp
"思"[1]

In [ ]:
%%cpp
"思"[2]

Finally:

In [ ]:
%%cpp
"思"[3]

The three characters before the null terminator are actually the three bytes that correspond to the [UTF-8 encoding of "思"](https://www.compart.com/en/unicode/U+601D). UTF-8 is a [variable-length encoding](https://en.wikipedia.org/wiki/Variable-length_code) designed to represent [Unicode characters](https://en.wikipedia.org/wiki/Unicode), including those that go beyond the ASCII character set.

For example, the emoticon below requires more bytes to encode:

In [ ]:
%%cpp
"😎"

A unicode symbol can also be encoded differently using [UTF-16](https://en.wikipedia.org/wiki/UTF-16):

In [ ]:
%%cpp
u"思"

In [ ]:
%%cpp
u"思"[0]

In [ ]:
%%cpp
u"思"[1]

In [ ]:
%%cpp
"\u601D" // escape code to enter a symbol using UTF-16 code

In [ ]:
%%hermes
In C++, how to return the number of symbols in a C string, given that the
symbols may be encoded using a variable-length code.

::::{admonition} Hermes
:class: dropdown

`strlen` counts **bytes**, not symbols — that is exactly why it returned 3 for `"思"` even though it is a single symbol. To count the symbols of a variable-length encoding, scan the byte sequence and use the *structure* of that encoding:


[](./count_symbols.ipynb)

- **UTF-8** (`"思"`): each symbol occupies 1–4 bytes. The first byte of a symbol is never a *continuation byte* (which has the bit pattern 0b10xxxxxx); all remaining bytes of the symbol are continuation bytes. So `utf8_symbols` counts the bytes `c` with `(static_cast<unsigned char>(c) & 0xC0) != 0x80`.
- **UTF-16** (`u"思"`): each unit is a `char16_t` (2 bytes). Most symbols occupy one unit, but a symbol outside the basic multilingual plane — such as 😎 — occupies a *surrogate pair*: a high surrogate (0xD800–0xDBFF) immediately followed by a low surrogate. So `utf16_symbols` counts the units, skipping the second unit of each pair.

The code above was compiled and run with g++ (`-std=c++23 -O2`) — the four lines below are its real output, not a sample:

```
"15": bytes=2, symbols=2
"思": bytes=3, symbols=1
"😎": bytes=4, symbols=1
u"思😎": code units=3, symbols=2
```

The last line is the sharpest contrast: `sizeof(u"思😎")/2 - 1` gives 3 (思 one unit + 😎 two units), while the number of symbols is 2 — the byte count with the terminator removed gets the answer wrong in **both** encodings for this string (7 vs 2 in UTF-8, 3 vs 2 in UTF-16), and only agrees by luck when every symbol is one byte (or one unit).

:::{caution}

Both counters assume the input is *well-formed* and do not validate it: a stray continuation byte or an unpaired surrogate skews the count. For production code, decode with a validating UTF-8/UTF-16 decoder (e.g., ICU or utf8proc) instead of hand-rolled bit tests.

:::

::::

[^hermes_response]: This is a sample response generated by the above prompt. Click me to expand the content.

There are [other types of string literals](https://en.cppreference.com/w/cpp/language/string_literal.html) in C++, but their support varies. For example, `strlen(u"1")` or `cout << u8"1";` will fail because these functions and operators do not handle wide or UTF-8 encoded strings directly. The exception is raw string literals, which ensure [WYSIWYG](https://en.wikipedia.org/wiki/WYSIWYG) (to some extent):

In [ ]:
%%cpp
cout << R"(
  ____  ____   ____   _____  _   ___  
 / ___|/ ___| |___ \ |___ / / | / _ \
| |    \___ \   __) |  |_ \ | || | | |
| |___  ___) | / __/  ___) || || |_| |
 \____||____/ |_____||____/ |_| \___/ 😎
)";

What does it mean to be a constant array?

Consider the following code:

In [ ]:
%%cpp
auto s="15";
s = "35";       // works
// s[0] = '1';  // fails

::::{caution} Why is it okay to change all the characters as in `s="35";` but not the first as in `s[0]=1;`?
:class: dropdown

`s` is indeed not a constant. It is a pointer that points to characters in read-only memory. 

- `s="35"` is allowed since it changes the value of the pointer to point to a new array of characters.
- `s[0]='3'` is not allowed since it attempts to rewrite a character in read-only memory.

::::

::::{exercise} 
:label: ex:auto_ref

Why the following code works even though `auto &b=1;` failed? Can I further assign `s` to another string as in `s="35";`?

::::

In [ ]:
%%cpp
auto &s="15";

YOUR ANSWER HERE